# ⚙️ Desafio 2 Comparando dados antes e depois da transformação
Durante a Aula 3, partimos dos dados brutos armazenados na camada Bronze e aplicamos transformações para construir a camada Silver.

Nesta etapa, trabalhamos com operações como ajustes de tipos, padronização de valores, tratamento de campos e organização dos dados para torná-los mais consistentes. 

Seu desafio é usar o Genie Code para ajudar você a comparar uma tabela da camada Bronze com sua versão correspondente na camada Silver.

A análise deve permitir verificar:

Quais colunas tiveram algum tipo de transformação? 

Os tipos de dados mudaram entre Bronze e Silver? 

Existem valores que foram padronizados ou tratados? 

Como os primeiros registros aparecem antes e depois da transformação?

O objetivo é perceber, na prática, como uma etapa de transformação muda a estrutura e a qualidade de uso dos dados dentro do pipeline.

✍️ Em Engenharia de Dados, transformar não significa apenas alterar valores. Significa preparar os dados para que eles sejam mais consistentes, compreensíveis e adequados para as próximas etapas.

In [0]:
# ============================================================
# CONFIGURACAO — alterar estas variaveis para comparar qualquer tabela
# ============================================================
catalog = 'voebem'
schema_bronze = 'bronze'
schema_silver = 'silver'

# Mapeamento silver -> bronze correspondente
# Para silver.empresas, usamos empresas_nacionais como referencia
# (a silver une nacionais + estrangeiras, mas o schema e identico)
bronze_map = {
    'vra': 'vra',
    'aerodromos': 'aerodromos',
    'codigos_operacao': 'codigos_operacao',
    'empresas': 'empresas_nacionais'
}

# >>> Altere apenas esta linha para trocar de tabela <<<
table_name = 'vra'
table_bronze = bronze_map.get(table_name, table_name)

print(f"Comparando: {catalog}.{schema_bronze}.{table_bronze}  ->  {catalog}.{schema_silver}.{table_name}")

display(spark.sql(f"""
WITH tb_bronze AS (
    SELECT column_name, data_type, ordinal_position
    FROM system.information_schema.columns
    WHERE table_catalog = '{catalog}'
      AND table_schema = '{schema_bronze}'
      AND table_name = '{table_bronze}'
),
tb_silver AS (
    SELECT column_name, data_type, ordinal_position
    FROM system.information_schema.columns
    WHERE table_catalog = '{catalog}'
      AND table_schema = '{schema_silver}'
      AND table_name = '{table_name}'
)
SELECT
    COALESCE(b.column_name, s.column_name) AS column_name,
    b.data_type AS tipo_bronze,
    s.data_type AS tipo_silver,
    CASE
        WHEN b.column_name IS NULL                       THEN 'COLUNA NOVA (nao existe na bronze)'
        WHEN s.column_name IS NULL                       THEN 'REMOVIDA OU RENOMEADA (nao existe na silver)'
        WHEN b.data_type != s.data_type                  THEN 'TIPO ALTERADO'
        ELSE 'OK'
    END AS status_transformacao
FROM tb_bronze b
FULL OUTER JOIN tb_silver s
    ON b.column_name = s.column_name
ORDER BY
    CASE
        WHEN b.column_name IS NULL      THEN 3
        WHEN s.column_name IS NULL      THEN 2
        WHEN b.data_type != s.data_type THEN 0
        ELSE 1
    END,
    COALESCE(b.ordinal_position, s.ordinal_position)
"""))

In [0]:
# 2. Comparacao de volumetria — a silver e espelho do bronze
# Se a diferenca nao for zero, algo foi filtrado ou duplicado
display(spark.sql(f"""
SELECT
    '{schema_bronze}.{table_bronze}'  AS tabela_bronze,
    '{schema_silver}.{table_name}'    AS tabela_silver,
    (SELECT COUNT(*) FROM {catalog}.{schema_bronze}.{table_bronze}) AS linhas_bronze,
    (SELECT COUNT(*) FROM {catalog}.{schema_silver}.{table_name})  AS linhas_silver,
    (SELECT COUNT(*) FROM {catalog}.{schema_bronze}.{table_bronze})
      - (SELECT COUNT(*) FROM {catalog}.{schema_silver}.{table_name}) AS diferenca
"""))

In [0]:
# 3. Para colunas que tiveram o tipo alterado, mostra valores de exemplo
# antes (bronze) e depois (silver) para evidenciar a transformacao aplicada

cols_diff = spark.sql(f"""
    WITH tb_bronze AS (
        SELECT column_name, data_type
        FROM system.information_schema.columns
        WHERE table_catalog = '{catalog}'
          AND table_schema = '{schema_bronze}'
          AND table_name = '{table_bronze}'
    ),
    tb_silver AS (
        SELECT column_name, data_type
        FROM system.information_schema.columns
        WHERE table_catalog = '{catalog}'
          AND table_schema = '{schema_silver}'
          AND table_name = '{table_name}'
    )
    SELECT b.column_name, b.data_type AS tipo_bronze, s.data_type AS tipo_silver
    FROM tb_bronze b
    INNER JOIN tb_silver s ON b.column_name = s.column_name
    WHERE b.data_type != s.data_type
    ORDER BY b.column_name
""").collect()

col_names = [r.column_name for r in cols_diff]

if col_names:
    print(f"Colunas com tipo alterado: {', '.join(col_names)}")
    print()

    for r in cols_diff:
        col = r.column_name
        print(f"--- {col}: {r.tipo_bronze} -> {r.tipo_silver} ---")
        print("BRONZE (5 exemplos):")
        display(spark.sql(f"""
            SELECT `{col}` FROM {catalog}.{schema_bronze}.{table_bronze}
            WHERE `{col}` IS NOT NULL
            LIMIT 5
        """))
        print("SILVER (5 exemplos):")
        display(spark.sql(f"""
            SELECT `{col}` FROM {catalog}.{schema_silver}.{table_name}
            WHERE `{col}` IS NOT NULL
            LIMIT 5
        """))
        print()
else:
    print("Nenhuma coluna com alteracao de tipo encontrada para esta tabela.")

In [0]:
# 4. Primeiros registros antes e depois da transformacao
# A silver e um espelho sem filtros, entao a ordem das linhas e preservada

print(f"=== BRONZE: {catalog}.{schema_bronze}.{table_bronze} (primeiros 5 registros) ===")
display(spark.sql(f"SELECT * FROM {catalog}.{schema_bronze}.{table_bronze} LIMIT 5"))

print(f"\n=== SILVER: {catalog}.{schema_silver}.{table_name} (primeiros 5 registros) ===")
display(spark.sql(f"SELECT * FROM {catalog}.{schema_silver}.{table_name} LIMIT 5"))